<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/benchmarks/benchmarks_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmarks Base Notebook

This notebook evaluates a Hugging Face model on lm-eval tasks, measures throughput and memory, and computes energy efficiency (Joules/token).

In [1]:
%pip install -q \
      "torch" \
      "transformers" \
      "accelerate" \
      "lm_eval" \
      "sentencepiece" \
      "langdetect" \
      "datasets" \
      "codecarbon" \
      "pandas"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.8/380.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.2 MB/s eta 0:00:00


In [2]:
import os
import json

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# The helper functions used in previous chapters are grouped in the utils.py file.
# We download it from the repository so the notebook is self-contained on Colab.
!wget -q https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/utils.py

import os
if os.path.exists('utils.py'):
    print("✅ utils.py downloaded successfully")
else:
    print("❌ Failed to download utils.py")

from utils import (
    model_evaluation,  # Benchmark evaluation with lm-eval
    clear_gpu_cache,
)

✅ utils.py downloaded successfully


In [4]:
import sys
import types

# Some environments have an optipfair/transformers version mismatch.
# The benchmark functions used in this notebook do not require optipfair directly.
sys.modules.setdefault("optipfair", types.ModuleType("optipfair"))

from utils import model_evaluation, measure_memory_allocation, measure_energy_consumption

print("Imported utils functions successfully.")

Imported utils functions successfully.


In [5]:
# -----------------------------
# Main configuration variables
# -----------------------------
MODEL_NAME = "meta-llama/Llama-3.2-3B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 64

BENCHMARK_TASKS = [
    "arc_easy",
    "winogrande",
    "hellaswag",
    "lambada_openai",
    "piqa",
]

BENCHMARK_LIMIT = None

print("Configuration:")
print(f"- MODEL_NAME: {MODEL_NAME}")
print(f"- DEVICE: {DEVICE}")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
print(f"- BENCHMARK_LIMIT (env BENCHMARK_LIMIT): {BENCHMARK_LIMIT}")
print(f"- BENCHMARK_TASKS: {BENCHMARK_TASKS}")

Configuration:
- MODEL_NAME: meta-llama/Llama-3.2-3B
- DEVICE: cuda
- BATCH_SIZE: 8
- MAX_NEW_TOKENS: 64
- BENCHMARK_LIMIT (env BENCHMARK_LIMIT): None
- BENCHMARK_TASKS: ['arc_easy', 'winogrande', 'hellaswag', 'lambada_openai', 'piqa']


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print("Model loaded successfully:")
print(f"- Name: {MODEL_NAME}")
print(f"- Device: {next(model.parameters()).device}")
print(f"- Dtype: {next(model.parameters()).dtype}")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded successfully:
- Name: meta-llama/Llama-3.2-3B
- Device: cuda:0
- Dtype: torch.bfloat16


In [7]:
benchmark_results = model_evaluation(
    model_obj=model,
    tokenizer=tokenizer,
    tasks=BENCHMARK_TASKS,
    device=DEVICE,
    limit=BENCHMARK_LIMIT,
    batch_size=BATCH_SIZE,
)

benchmark_rows = []
for task_name, metrics in benchmark_results.items():
    row = {"task": task_name}
    row.update(metrics)
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)
print("\nBenchmark results:")
display(benchmark_df)

Starting lm-eval on model 'meta-llama/Llama-3.2-3B' for tasks: ['arc_easy', 'winogrande', 'hellaswag', 'lambada_openai', 'piqa']



Tasks grouped by few-shot: {0: ['arc_easy', 'winogrande', 'hellaswag', 'lambada_openai', 'piqa']} (full dataset)
Task-level few-shot config: {'arc_easy': 0, 'winogrande': 0, 'hellaswag': 0, 'lambada_openai': 0, 'piqa': 0}

Evaluating 5 task(s) with 0-shot learning...


README.md: 0.00B [00:00, ?B/s]

ARC-Easy/train-00000-of-00001.parquet:   0%|          | 0.00/331k [00:00<?, ?B/s]

ARC-Easy/test-00000-of-00001.parquet:   0%|          | 0.00/346k [00:00<?, ?B/s]

...

Running loglikelihood requests: 100%|██████████| 61032/61032 [5:07:07<00:00,  3.31it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:13<00:00,  7.68it/s]



Benchmark results:


,task,accuracy,acc_norm,perplexity
0,arc_easy,0.7449,0.7201,NaN
1,hellaswag,0.5581,0.7410,NaN
2,lambada_openai,0.6978,NaN,3.88
3,piqa,0.7633,0.7802,NaN
4,winogrande,0.6930,NaN,NaN


In [8]:
PERFORMANCE_PROMPTS = [
    "Explain what model pruning is in one paragraph.",
    "Summarize why KV cache improves autoregressive generation speed.",
    "List three tradeoffs between model quality and inference efficiency.",
]

memory_perf_runs = []
for prompt in PERFORMANCE_PROMPTS:
    metrics = measure_memory_allocation(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    metrics["prompt"] = prompt
    memory_perf_runs.append(metrics)

memory_perf_df = pd.DataFrame(memory_perf_runs)
performance_summary = {
    "throughput_tokens_s_avg": float(memory_perf_df["throughput_tokens_s"].mean()),
    "throughput_tokens_s_min": float(memory_perf_df["throughput_tokens_s"].min()),
    "throughput_tokens_s_max": float(memory_perf_df["throughput_tokens_s"].max()),
    "static_vram_mb_avg": float(memory_perf_df["static_vram_mb"].mean()),
    "dynamic_delta_mb_avg": float(memory_perf_df["dynamic_delta_mb"].mean()),
}

print("Per-prompt throughput and memory:")
display(memory_perf_df[["prompt", "static_vram_mb", "dynamic_delta_mb", "throughput_tokens_s"]])

print("\nAggregated throughput and memory:")
for k, v in performance_summary.items():
    print(f"- {k}: {v}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Per-prompt throughput and memory:


,prompt,static_vram_mb,dynamic_delta_mb,throughput_tokens_s
0,Explain what model pruning is in one paragraph.,6136.46,11.82,15.42
1,Summarize why KV cache improves autoregressive...,6136.46,12.28,19.67
2,List three tradeoffs between model quality and...,6136.46,11.98,20.40



Aggregated throughput and memory:
- throughput_tokens_s_avg: 18.496666666666666
- throughput_tokens_s_min: 15.42
- throughput_tokens_s_max: 20.4
- static_vram_mb_avg: 6136.46
- dynamic_delta_mb_avg: 12.026666666666666


In [9]:
ENERGY_PROMPTS = [
    "Describe the main difference between pruning and quantization.",
    "Write a short explanation of why batch size affects throughput.",
    "Give two examples of NLP benchmark tasks and what they measure.",
    "Explain in simple words what Joules per token means.",
]

class PromptDataset(Dataset):
    def __init__(self, prompts, tokenizer, max_length=256):
        self.enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
        }

energy_dataset = PromptDataset(ENERGY_PROMPTS, tokenizer)
energy_loader = DataLoader(energy_dataset, batch_size=1, shuffle=False)

In [10]:
energy_results = measure_energy_consumption(
    model=model,
    tokenizer=tokenizer,
    data_source=energy_loader,
    idle_power_watts=None,
    num_runs=1,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=len(energy_dataset),
)

energy_joules = float(energy_results["energy_net_kwh"] * 3_600_000)
joules_per_token = float(energy_results["efficiency_joules_per_token"])

print("Energy metrics:")
print(f"- energy_net_kwh: {energy_results['energy_net_kwh']}")
print(f"- energy_joules: {energy_joules}")
print(f"- total_tokens: {energy_results['total_tokens']}")
print(f"- duration_sec: {energy_results['duration_sec']}")
print(f"- efficiency_joules_per_token (J/token): {joules_per_token}")
print(f"- co2_emissions_kg: {energy_results['co2_emissions_kg']}")

[codecarbon WARNING @ 19:59:27] Multiple instances of codecarbon are allowed to run at the same time.


   ⚙️ Auto-calibrating idle power (30s)...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


   ✓ Measured idle power: 95.82W
🌍 Measuring energy on 4 samples (1 runs each)...
   🔥 Performing GPU Warm-up...


Energy measurement: 100%|██████████| 4/4 [00:11<00:00,  3.00s/it]

Energy metrics:
- energy_net_kwh: 0.00010626445370529584
- energy_joules: 382.552033339065
- total_tokens: 256
- duration_sec: 12.012784719467163
- efficiency_joules_per_token (J/token): 1.4943438802307227
- co2_emissions_kg: 0.00012162972916230888


In [11]:
final_summary = {
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "DEVICE": DEVICE,
        "BATCH_SIZE": BATCH_SIZE,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "BENCHMARK_LIMIT": BENCHMARK_LIMIT,
        "BENCHMARK_TASKS": BENCHMARK_TASKS,
    },
    "benchmarks": benchmark_results,
    "performance_memory": {
        "per_prompt": memory_perf_runs,
        "summary": performance_summary,
    },
    "energy": {
        **energy_results,
        "energy_joules": energy_joules,
        "joules_per_token": joules_per_token,
    },
}

print("================ FINAL SUMMARY ================")
print(json.dumps(final_summary, indent=2, default=str))

================ FINAL SUMMARY ================
{
  "config": {
    "MODEL_NAME": "meta-llama/Llama-3.2-3B",
    "DEVICE": "cuda",
    "BATCH_SIZE": 8,
    "MAX_NEW_TOKENS": 64,
    "BENCHMARK_LIMIT": null,
    "BENCHMARK_TASKS": [
      "arc_easy",
      "winogrande",
      "hellaswag",
      "lambada_openai",
      "piqa"
    ]
  },
  "benchmarks": {
    "arc_easy": {
      "accuracy": "0.7449",
      "acc_norm": "0.7201"
    },
    "hellaswag": {
      "accuracy": "0.5581",
      "acc_norm": "0.7410"
    },
    "lambada_openai": {
      "perplexity": "3.88",
      "accuracy": "0.6978"
    },
    "piqa": {
      "accuracy": "0.7633",
      "acc_norm": "0.7802"
    },
    "winogrande": {
      "accuracy": "0.6930"
    }
  },
  "performance_memory": {
    "per_prompt": [
      {
        "static_vram_mb": 6136.46,
        "dynamic_delta_mb": 11.82,
        "throughput_tokens_s": 15.42,
        "generated_text": " What is the difference between model pruning and feature pruning? What is 

In [12]:
gpu_info = {}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_info = {
        "gpu_name": props.name,
        "total_memory_gb": round(props.total_memory / (1024**3), 2),
        "compute_capability": f"{props.major}.{props.minor}",
        "cuda_version": torch.version.cuda,
        "reserved_memory_mb": torch.cuda.memory_reserved(0) / (1024**2)
    }
else:
    gpu_info = {"status": "No CUDA device detected"}

# Añadimos esta información al diccionario final
final_summary["gpu_hardware"] = gpu_info

In [13]:
import json

# Limpiamos el nombre del modelo para que sea un nombre de archivo válido
# (reemplazamos la barra '/' por un guion bajo)
safe_model_name = MODEL_NAME.replace("/", "_")
file_name = f"{safe_model_name}_summary.json"

# Guardamos el diccionario final_summary en el archivo
with open(file_name, "w") as f:
    json.dump(final_summary, f, indent=2, default=str)

print(f"Resumen guardado correctamente en: {file_name}")

Resumen guardado correctamente en: meta-llama_Llama-3.2-3B_summary.json
